# Publication notebook: pair-aware EDA

- Purpose: exploratory summaries for the retained pair-aware feature dataset.
- Required inputs: `data/processed/all_reads.tsv`.
- Required models: none.
- Required external tools: none.
- Expected outputs: EDA figures and feature-summary tables under `results/`.
- Publication output: descriptive feature-distribution figures and summary tables.
- Reproduction status: path-normalized in this remediation pass; not rerun here.


In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 120

df = pd.read_csv("../data/processed/all_reads.tsv", sep="\t")

print("Shape:", df.shape)
print("\nLabel proportions:")
print(df["label"].value_counts(normalize=True))
print("\nLabel counts:")
print(df["label"].value_counts())

In [ ]:
print(df.columns.tolist())
df.head()

In [ ]:
features_to_plot = [
    "sa_count",
    "softclip_left",
    "softclip_right",
    "kmer_js_divergence",
    "kmer_cosine_diff",
    "microhomology_length",
]

summary_list = []

for feat in features_to_plot:
    stats = (
        df.groupby("label")[feat]
        .describe(percentiles=[0.25, 0.5, 0.75])
        .reset_index()
    )
    stats.insert(0, "feature", feat)
    summary_list.append(stats)

summary_df = pd.concat(summary_list, ignore_index=True)
summary_df

In [ ]:
out_path = Path("feature_summary_by_class_PAIR.tsv")
summary_df.to_csv(out_path, sep="\t", index=False)
print("Saved:", out_path.resolve())

In [ ]:
Path("../results/figures/notebook_exports").mkdir(parents=True, exist_ok=True)

features_to_plot = [
    "sa_count",
    "softclip_left",
    "softclip_right",
    "kmer_js_divergence",
    "kmer_cosine_diff",
    "microhomology_length",
]

for feat in features_to_plot:
    plt.figure(figsize=(5, 4))

    for label, color, name in [
        (0, "tab:blue", "clean"),
        (1, "tab:orange", "chimeric"),
    ]:
        subset = df.loc[df["label"] == label, feat]
        plt.hist(
            subset,
            bins=40,
            density=True,
            alpha=0.5,
            label=name,
        )

    plt.xlabel(feat)
    plt.ylabel("Density")
    plt.title(f"{feat} by class")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"figures/eda_{feat}_PAIR.png", dpi=300, bbox_inches="tight")
    plt.close()

print("Saved histogram figures to ../results/figures/notebook_exports/")

In [ ]:
numeric_df = df.select_dtypes(include=["number"])
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("label").mean(numeric_only=True).T.sort_values(by=0, ascending=False)

In [ ]:
df2 = df.copy()
df2["class"] = df2["label"].map({0: "clean", 1: "chimeric"})

numeric_cols = df2.select_dtypes(include=["number"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "label"]

rows = []
for feat in numeric_cols:
    for cls_name, group in df2.groupby("class"):
        s = group[feat]
        rows.append(
            {
                "feature": feat,
                "class": cls_name,
                "mean": s.mean(),
                "std": s.std(),
                "median": s.median(),
                "q1": s.quantile(0.25),
                "q3": s.quantile(0.75),
                "iqr": s.quantile(0.75) - s.quantile(0.25),
                "min": s.min(),
                "max": s.max(),
                "n": s.count(),
            }
        )

summary_stats = pd.DataFrame(rows).sort_values(["feature", "class"])
summary_stats.head()

In [ ]:
out_dir = Path("../results")
out_dir.mkdir(parents=True, exist_ok=True)

summary_path = out_dir / "feature_summary_by_class_PAIR.tsv"
summary_stats.to_csv(summary_path, sep="\t", index=False)

print("Saved:", summary_path.resolve())

In [ ]:
df3 = df.copy()
df3["class"] = df3["label"].map({0: "clean", 1: "chimeric"})

numeric_cols = [
    c for c in df3.columns
    if c not in ["label", "class"] and pd.api.types.is_numeric_dtype(df3[c])
]

print("Number of numeric features:", len(numeric_cols))
print("Example features:", numeric_cols[:10])

os.makedirs("../results/figures/notebook_exports/boxplots_PAIR", exist_ok=True)

for feat in numeric_cols:
    plt.figure(figsize=(5, 4))
    sns.boxplot(
        data=df3,
        x="class",
        y=feat,
        showfliers=True,
    )
    plt.xlabel("Class")
    plt.ylabel(feat)
    plt.title(f"{feat} by class (boxplot)")
    plt.tight_layout()
    plt.savefig(f"figures/boxplots_PAIR/box_{feat}.png", dpi=300, bbox_inches="tight")
    plt.close()

print("Saved boxplots to ../results/figures/notebook_exports/boxplots_PAIR/")